In [2]:
import warnings

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import pickle
warnings.filterwarnings('ignore')


In [3]:
df=pd.read_csv('C:\\Users\\Venkata Lakshmi\\Downloads\\loan_prediction.csv')    
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [12]:
df.isnull().mean()

Gender               0.0
Married              0.0
Dependents           0.0
Education            0.0
Self_Employed        0.0
ApplicantIncome      0.0
CoapplicantIncome    0.0
LoanAmount           0.0
Loan_Amount_Term     0.0
Credit_History       0.0
Property_Area        0.0
Loan_Status          0.0
dtype: float64

In [15]:
# Loan_ID కాలమ్ మనకి మోడల్ ట్రైనింగ్ కి అవసరం లేదు కాబట్టి తీసేస్తున్నాం
if 'Loan_ID' in df.columns:
    df = df.drop(columns=['Loan_ID'])

# Dependents కాలమ్‌లో '3+' ని 3గా మార్చడం
df['Dependents'] = df['Dependents'].replace('3+', 3).astype(int)

# మిగిలిన టెక్స్ట్ కాలమ్స్‌ని నంబర్లలోకి మ్యాపింగ్ చేయడం
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['Married'] = df['Married'].map({'Yes': 1, 'No': 0})
df['Education'] = df['Education'].map({'Graduate': 1, 'Not Graduate': 0})
df['Self_Employed'] = df['Self_Employed'].map({'Yes': 1, 'No': 0})
df['Property_Area'] = df['Property_Area'].map({'Rural': 0, 'Semiurban': 1, 'Urban': 2})
df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})

# మార్పులు సరిగ్గా జరిగాయో లేదో చూడటానికి
df.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,1,0,0,1,0,5849,0.0,128.0,360.0,1,2,1
1,1,1,1,1,0,4583,1508.0,128.0,360.0,1,0,0
2,1,1,0,1,1,3000,0.0,66.0,360.0,1,2,1
3,1,1,0,0,0,2583,2358.0,120.0,360.0,1,2,1
4,1,0,0,1,0,6000,0.0,141.0,360.0,1,2,1


In [17]:
# Target Variable (Loan_Status) ని y లోకి, మిగిలిన ఫీచర్లను X లోకి తీస్కోవాలి
X = df.drop(columns=['Loan_Status'])
y = df['Loan_Status']

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [19]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import pickle

# 1. నాలుగు మోడల్స్‌ను క్రియేట్ చేయడం
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

# 2. అన్ని మోడల్స్‌ని ట్రైన్ చేసి Accuracy కంపేర్ చేయడం
print("📊 Model Accuracy Comparison:\n" + "-"*35)
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name:<15} Accuracy: {acc*100:.2f}%")

# 3. అత్యధిక Accuracy ఉన్న మోడల్‌ను ఫైల్‌గా సేవ్ చేయడం (model.pkl)
best_model_name = max(results, key=results.get)
best_model = models[best_model_name]

print("\n" + "="*45)
print(f"🏆 Best Model: {best_model_name} with {results[best_model_name]*100:.2f}% accuracy!")

with open('model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print("✅ Best Model saved as 'model.pkl' for Flask App!")

📊 Model Accuracy Comparison:
-----------------------------------
Decision Tree   Accuracy: 68.29%
Random Forest   Accuracy: 76.42%
KNN             Accuracy: 57.72%
XGBoost         Accuracy: 75.61%

🏆 Best Model: Random Forest with 76.42% accuracy!
✅ Best Model saved as 'model.pkl' for Flask App!
